In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pycocotools.coco import COCO
import os

nc = 80
def convert_bbox_to_yolo(image_width, image_height, bbox):
    x, y, w, h = bbox
    x_center = (x + w / 2) / image_width
    y_center = (y + h / 2) / image_height
    w = w / image_width
    h = h / image_height
    return x_center, y_center, w, h

def convert_coco_to_yolo_per_folder(dataset_root, billboard_folders, subset="train", nc=80):
    for folder in billboard_folders:
        ann_path = os.path.join(dataset_root, folder, "annotations", subset, "coco_annotations.json")
        coco = COCO(ann_path)

        # Create mapping: COCO category ID -> index 0–79
        cats = coco.loadCats(coco.getCatIds())
        coco_id_to_yolo_id = {cat['id']: i for i, cat in enumerate(sorted(cats, key=lambda c: c['id']))}

        label_dir = os.path.join(dataset_root, folder, "labels", subset)
        os.makedirs(label_dir, exist_ok=True)

        for img_id in coco.getImgIds():
            img = coco.loadImgs([img_id])[0]
            anns = coco.loadAnns(coco.getAnnIds(imgIds=[img_id]))

            if not anns:
                continue

            label_lines = []
            for ann in anns:
                if 'bbox' not in ann or ann['category_id'] not in coco_id_to_yolo_id:
                    continue

                yolo_bbox = convert_bbox_to_yolo(img["width"], img["height"], ann["bbox"])
                class_id = coco_id_to_yolo_id[ann["category_id"]]  #  mapped class ID
                label_lines.append(f"{class_id} {' '.join(map(str, yolo_bbox))}")

            label_filename = img["file_name"].rsplit(".", 1)[0] + ".txt"
            label_path = os.path.join(label_dir, label_filename)

            with open(label_path, "w") as f:
                f.write("\n".join(label_lines))

        print(f"{folder}: YOLO labels saved to {label_dir}")


In [ ]:
def create_image_txt_list(dataset_root, billboard_folders, subset="train"):
    txt_path = os.path.join(dataset_root, f"{subset}.txt")
    with open(txt_path, "w") as f:
        for bb in billboard_folders:
            image_dir = os.path.join(dataset_root, bb, "images", subset)
            if not os.path.exists(image_dir):
                print(f"Skipping missing image dir: {image_dir}")
                continue

            for img_name in sorted(os.listdir(image_dir)):
                if img_name.endswith((".jpg", ".jpeg", ".png")):
                    img_path = os.path.join(image_dir, img_name)
                    f.write(img_path + "\n")
    print(f"{subset}.txt created at: {txt_path}")


In [ ]:

dataset_root = "/content/drive/MyDrive/2dod/2dod"
billboards = ["billboard01", "billboard02", "billboard03", "billboard04",
              "billboard05", "billboard06", "billboard07", "billboard08", "billboard09"]

# Convert both train and val annotations
convert_coco_to_yolo_per_folder(dataset_root, billboards, subset="train")
convert_coco_to_yolo_per_folder(dataset_root, billboards, subset="val")

# Create train.txt and val.txt
create_image_txt_list(dataset_root, billboards, subset="train")
create_image_txt_list(dataset_root, billboards, subset="val")


In [ ]:
data_yaml_path = "/content/drive/MyDrive/2dod/2dod/data.yaml"

data_yaml_content = """\
train: /content/drive/MyDrive/2dod/2dod/train.txt
val: /content/drive/MyDrive/2dod/2dod/val.txt
nc: 80
names:
  0: person
  1: bicycle
  2: car
  3: motorcycle
  4: airplane
  5: bus
  6: train
  7: truck
  8: boat
  9: traffic light
  10: fire hydrant
  11: stop sign
  12: parking meter
  13: bench
  14: bird
  15: cat
  16: dog
  17: horse
  18: sheep
  19: cow
  20: elephant
  21: bear
  22: zebra
  23: giraffe
  24: backpack
  25: umbrella
  26: handbag
  27: tie
  28: suitcase
  29: frisbee
  30: skis
  31: snowboard
  32: sports ball
  33: kite
  34: baseball bat
  35: baseball glove
  36: skateboard
  37: surfboard
  38: tennis racket
  39: bottle
  40: wine glass
  41: cup
  42: fork
  43: knife
  44: spoon
  45: bowl
  46: banana
  47: apple
  48: sandwich
  49: orange
  50: broccoli
  51: carrot
  52: hot dog
  53: pizza
  54: donut
  55: cake
  56: chair
  57: couch
  58: potted plant
  59: bed
  60: dining table
  61: toilet
  62: tv
  63: laptop
  64: mouse
  65: remote
  66: keyboard
  67: cell phone
  68: microwave
  69: oven
  70: toaster
  71: sink
  72: refrigerator
  73: book
  74: clock
  75: vase
  76: scissors
  77: teddy bear
  78: hair drier
  79: toothbrush
"""

with open(data_yaml_path, "w") as f:
    f.write(data_yaml_content)

print(f" data.yaml saved at: {data_yaml_path}")


Verify annotation conversion.

In [ ]:
# Visualize YOLO bounding boxes (from train.txt) using full COCO class list

import cv2
import os
import random
import matplotlib.pyplot as plt

# Full COCO 80 class names
class_names = [
    "person", "bicycle", "car", "motorcycle", "airplane", "bus", "train", "truck", "boat", "traffic light",
    "fire hydrant", "stop sign", "parking meter", "bench", "bird", "cat", "dog", "horse", "sheep", "cow",
    "elephant", "bear", "zebra", "giraffe", "backpack", "umbrella", "handbag", "tie", "suitcase", "frisbee",
    "skis", "snowboard", "sports ball", "kite", "baseball bat", "baseball glove", "skateboard", "surfboard",
    "tennis racket", "bottle", "wine glass", "cup", "fork", "knife", "spoon", "bowl", "banana", "apple",
    "sandwich", "orange", "broccoli", "carrot", "hot dog", "pizza", "donut", "cake", "chair", "couch",
    "potted plant", "bed", "dining table", "toilet", "tv", "laptop", "mouse", "remote", "keyboard", "cell phone",
    "microwave", "oven", "toaster", "sink", "refrigerator", "book", "clock", "vase", "scissors", "teddy bear",
    "hair drier", "toothbrush"
]

def visualize_yolo_label(img_path, label_path, class_names):
    img = cv2.imread(img_path)
    if img is None:
        print(f"Image not found: {img_path}")
        return

    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w, _ = img.shape

    if not os.path.exists(label_path):
        print(f"Label file not found: {label_path}")
        return

    with open(label_path, "r") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) != 5:
                continue
            class_id = int(parts[0])
            x_c, y_c, bw, bh = map(float, parts[1:])

            # Convert YOLO (normalized) to pixel box
            x1 = int((x_c - bw / 2) * w)
            y1 = int((y_c - bh / 2) * h)
            x2 = int((x_c + bw / 2) * w)
            y2 = int((y_c + bh / 2) * h)

            cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
            label = class_names[class_id] if class_id < len(class_names) else str(class_id)
            cv2.putText(img, label, (x1, y1 - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

    plt.figure(figsize=(10, 10))
    plt.imshow(img)
    plt.axis("off")
    plt.show()

#  Load 5 random samples from train.txt
dataset_root = "/content/drive/MyDrive/2dod/2dod"
train_txt = os.path.join(dataset_root, "train.txt")

with open(train_txt, "r") as f:
    image_paths = [line.strip() for line in f if line.strip()]

for img_path in random.sample(image_paths, 5):
    label_path = img_path.replace("/images/", "/labels/").rsplit(".", 1)[0] + ".txt"
    visualize_yolo_label(img_path, label_path, class_names)



In [ ]:
!pip install ultralytics

In [ ]:
%pip install -U ultralytics


In [ ]:
from ultralytics import YOLO

# Train YOLOv8 on Custom Dataset


In [ ]:
model = YOLO("yolov8n.pt")

# Continue training from that checkpoint
model.train(
    data="/content/drive/MyDrive/2dod/2dod/data.yaml",
    epochs=200,
    imgsz=640,
    batch=8,
    save_period=100,
)

In [ ]:
import shutil
import os
import glob

# Define paths
run_folder = "/content/runs/detect/train"
weights_folder = os.path.join(run_folder, "weights")
drive_folder = "/content/drive/MyDrive/yolo_trained_models/2dod_checkpoints/"

# Create the Drive folder if it doesn't exist
os.makedirs(drive_folder, exist_ok=True)

# Copy all YOLO checkpoint files (.pt) to Google Drive
checkpoints = glob.glob(os.path.join(run_folder, "*.pt")) + glob.glob(os.path.join(weights_folder, "*.pt"))
for ckpt in checkpoints:
    shutil.copy(ckpt, drive_folder)

print(f" All weights copied to Google Drive → {drive_folder}")


In [ ]:
import os, glob, shutil
import pandas as pd

# Find the latest results.csv
latest = sorted(
    glob.glob('runs/detect/train*/results.csv'),
    key=os.path.getmtime
)[-1]

# Load it if needed
df = pd.read_csv(latest)
print(f"Using: {latest}")

# Copy to Drive
shutil.copy(latest, "/content/drive/MyDrive/yolo_trained_models/2dod_checkpoints/results.csv")
print("✅ CSV copied to Drive.")



In [ ]:
import os, glob

print("Runs:", glob.glob("/content/runs/detect/*"))
for d in sorted(glob.glob("/content/runs/detect/*")):
    print(d, "->", "weights exists?" , os.path.isdir(os.path.join(d, "weights")))
    print("  weights:", glob.glob(os.path.join(d, "weights", "*.pt")))
    print("  results:", glob.glob(os.path.join(d, "results.csv")))


In [ ]:
import os, glob, shutil

src_weights = "/content/runs/detect/train2/weights"
src_results = "/content/runs/detect/train2/results.csv"
dst = "/content/drive/MyDrive/yolo_trained_models/2dod_checkpoints/"

os.makedirs(dst, exist_ok=True)

# copy weights
for f in glob.glob(os.path.join(src_weights, "*.pt")):
    shutil.copy(f, dst)
    print("copied:", os.path.basename(f))

# copy results.csv
shutil.copy(src_results, os.path.join(dst, "results.csv"))
print("copied: results.csv")

print("✅ Drive folder now contains:", glob.glob(dst + "*"))


In [ ]:
import os, glob
import pandas as pd
import matplotlib.pyplot as plt

# 1) Find latest results.csv (train, train2, train3...)
results_path = sorted(
    glob.glob("/content/runs/detect/train*/results.csv"),
    key=os.path.getmtime
)[-1]

print("Using:", results_path)

df = pd.read_csv(results_path)

# 2) Helper: find the first matching column name from candidates
def pick_col(candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

# Typical Ultralytics YOLO column names (varies by version)
train_box = pick_col(["train/box_loss", "train/box_loss(B)"])
train_cls = pick_col(["train/cls_loss", "train/cls_loss(B)"])
val_box   = pick_col(["val/box_loss", "val/box_loss(B)"])
val_cls   = pick_col(["val/cls_loss", "val/cls_loss(B)"])

map50     = pick_col(["metrics/mAP50(B)", "metrics/mAP50", "metrics/mAP_0.5"])
map5095   = pick_col(["metrics/mAP50-95(B)", "metrics/mAP50-95", "metrics/mAP_0.5:0.95"])

# 3) Build plots
epochs = range(1, len(df) + 1)

plt.figure(figsize=(16, 8))

# --- Top: losses ---
plt.subplot(2, 1, 1)
plt.title("Losses over Epochs")
if train_box: plt.plot(epochs, df[train_box], label="Train Box Loss")
if train_cls: plt.plot(epochs, df[train_cls], label="Train Cls Loss")
if val_box:   plt.plot(epochs, df[val_box],   label="Val Box Loss")
if val_cls:   plt.plot(epochs, df[val_cls],   label="Val Cls Loss")
plt.ylabel("Loss")
plt.grid(True, alpha=0.3)
plt.legend()

# --- Bottom: mAP ---
plt.subplot(2, 1, 2)
plt.title("mAP Performance over Epochs")
if map50:   plt.plot(epochs, df[map50],   label="mAP@50")
if map5095: plt.plot(epochs, df[map5095], label="mAP@50-95")
plt.xlabel("Epoch")
plt.ylabel("mAP")
plt.grid(True, alpha=0.3)
plt.legend()

plt.tight_layout()

# 4) Save next to results.csv
out_png = os.path.join(os.path.dirname(results_path), "training_curves.png")
plt.savefig(out_png, dpi=200)
print("✅ Saved plot to:", out_png)

plt.show()


In [ ]:
from ultralytics import YOLO

# Load the best checkpoint from Google Drive
model = YOLO("/content/drive/MyDrive/yolo_trained_models/2dod_checkpoints/best.pt")

# Now you can use it for prediction, validation, etc.
#results = model.predict(source="path/to/your/image_or_video.jpg")  # or .mp4, folder, etc.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Load the CSV
df = pd.read_csv(latest)

# Initialize lists to collect values
epochs = []
train_box_loss = []
train_cls_loss = []
val_box_loss = []
val_cls_loss = []
map50 = []
map5095 = []

# Collect data while printing
for i, row in df.iterrows():
    epochs.append(int(row['epoch']))
    train_box_loss.append(row['train/box_loss'])
    train_cls_loss.append(row['train/cls_loss'])
    val_box_loss.append(row['val/box_loss'])
    val_cls_loss.append(row['val/cls_loss'])
    map50.append(row['metrics/mAP50(B)'])
    map5095.append(row['metrics/mAP50-95(B)'])

    print(f"Epoch {int(row['epoch'])}: "
          f"Train Box Loss={row['train/box_loss']:.3f}, "
          f"Train Cls Loss={row['train/cls_loss']:.3f}, "
          f"Val Box Loss={row['val/box_loss']:.3f}, "
          f"Val Cls Loss={row['val/cls_loss']:.3f}, "
          f"mAP50={row['metrics/mAP50(B)']:.3f}, "
          f"mAP50-95={row['metrics/mAP50-95(B)']:.3f}")

# Now plot them
plt.figure(figsize=(14, 8))

# Losses
plt.subplot(2, 1, 1)
plt.plot(epochs, train_box_loss, label='Train Box Loss')
plt.plot(epochs, train_cls_loss, label='Train Cls Loss')
plt.plot(epochs, val_box_loss, label='Val Box Loss')
plt.plot(epochs, val_cls_loss, label='Val Cls Loss')
plt.ylabel('Loss')
plt.title('Losses over Epochs')
plt.legend()
plt.grid(True)

# mAP
plt.subplot(2, 1, 2)
plt.plot(epochs, map50, label='mAP@50')
plt.plot(epochs, map5095, label='mAP@50-95')
plt.xlabel('Epoch')
plt.ylabel('mAP')
plt.title('mAP Performance over Epochs')
plt.legend()
plt.grid(True)

# Save plots to Drive
plt.tight_layout()
plt.savefig("/content/drive/MyDrive/yolo_trained_models/2dod_checkpoints/metrics_summary.png")
print("plot saved to Drive.")
plt.show()


In [ ]:
import shutil
import os
import glob

# Automatically find the latest trainX folder
results_csv = sorted(
    glob.glob("/content/runs/detect/train*/results.csv"),
    key=os.path.getmtime
)[-1]  # get most recent CSV file

# Define save location
dst = "/content/drive/MyDrive/yolo_trained_models/2dod_checkpoints/results.csv"

# Copy it
shutil.copy(results_csv, dst)
print(f"✅ CSV copied from {results_csv} to {dst}")


In [ ]:
import os
import glob
import matplotlib.pyplot as plt

# Auto-detect the most recent training run that contains results
train_dirs = sorted(glob.glob("runs/detect/train*/"), key=os.path.getmtime, reverse=True)

# Pick the most recent directory that contains at least one plot image
train_dir = None
for d in train_dirs:
    if any(os.path.exists(os.path.join(d, f)) for f in ["F1_curve.png", "results.png", "labels.jpg"]):
        train_dir = d
        break

if train_dir is None:
    raise FileNotFoundError("No YOLO training run with result images found.")

print(f"Using training results from: {train_dir}")

# Filenames of training result plots
train_images = [
    "BoxF1_curve.png",
    "BoxP_curve.png",
    "BoxR_curve.png",
    "confusion_matrix.png",
    "labels.jpg",
    "results.png",
]


# Grid setup
num_images = len(train_images)
cols = 2
rows = (num_images + cols - 1) // cols  # ceiling division

# Bigger figure size (adjust if needed)
fig, axes = plt.subplots(rows, cols, figsize=(16, rows * 6))
axes = axes.flatten()

# Display each image in the grid
for i, img_name in enumerate(train_images):
    img_path = os.path.join(train_dir, img_name)
    if os.path.exists(img_path):
        print(f"Displaying: {img_name}")
        img = plt.imread(img_path)
        axes[i].imshow(img)
        axes[i].axis('off')
        axes[i].set_title(img_name, fontsize=14)
    else:
        print(f"File not found: {img_name}")
        axes[i].axis('off')

# Hide any unused subplots
for j in range(i + 1, len(axes)):
    axes[j].axis('off')

plt.tight_layout()
plt.show()


In [ ]:
import os
import glob
from IPython.display import Image, display

#  Auto-find latest training folder that has val images
train_dirs = sorted(glob.glob("runs/detect/train*/"), key=os.path.getmtime, reverse=True)

val_dir = None
for d in train_dirs:
    if any(os.path.exists(os.path.join(d, f)) for f in ["val_batch0_pred.jpg", "F1_curve.png"]):
        val_dir = d
        break

if val_dir is None:
    raise FileNotFoundError("No validation images found in any runs/detect/train*/ folder.")

print(f" Using validation results from: {val_dir}")

val_images = [
    "BoxF1_curve.png",
    "BoxP_curve.png",
    "BoxR_curve.png",
    "confusion_matrix.png",
    "labels.jpg",
    "results.png",
]
# Display images
for img_name in val_images:
    img_path = os.path.join(val_dir, img_name)
    if os.path.exists(img_path):
        print(f" Displaying: {img_name}")
        display(Image(filename=img_path))
    else:
        print(f"File not found: {img_name}")


In [ ]:
def create_image_txt_list(dataset_root, billboards, subset="test_nopatch"):
    """
    Appends all image paths from specified billboards and subset into a single txt list.
    """
    output_txt = os.path.join(dataset_root, f"{subset}.txt")

    with open(output_txt, "a") as f:
        for billboard in billboards:
            image_folder = os.path.join(dataset_root, billboard, "images", subset)
            if not os.path.exists(image_folder):
                print(f"⚠️ Warning: {image_folder} does not exist.")
                continue

            images = sorted([
                os.path.join(image_folder, file)
                for file in os.listdir(image_folder)
                if file.endswith((".jpg", ".png"))
            ])
            for img_path in images:
                f.write(img_path + "\n")


In [ ]:
def convert_all_test_sets(dataset_root, billboards, test_subfolders):
    for test_name in test_subfolders:
        # Clear the final txt once per test set
        txt_path = os.path.join(dataset_root, f"{test_name}.txt")
        open(txt_path, 'w').close()

        for billboard in billboards:
            print(f"\nProcessing {billboard} / {test_name}")
            convert_coco_to_yolo_per_folder(dataset_root, [billboard], subset=test_name)

        # After processing all billboards for this test set, collect image paths
        create_image_txt_list(dataset_root, billboards, subset=test_name)


In [ ]:
dataset_root = "/content/drive/MyDrive/2dod/2dod"

billboards = [
    "billboard01", "billboard02", "billboard03", "billboard04",
    "billboard05", "billboard06", "billboard07", "billboard08", "billboard09"
]

test_folders = ["test_nopatch", "test_frcnn", "test_random", "test_retinanet"]

convert_all_test_sets(dataset_root, billboards, test_folders)


In [ ]:
yaml_content = """\
path: /content/drive/MyDrive/2dod/2dod/
train: /content/drive/MyDrive/2dod/2dod/train.txt
val: /content/drive/MyDrive/2dod/2dod/val.txt
test: test_nopatch.txt

names:
  0: person
  1: bicycle
  2: car
  3: motorcycle
  4: airplane
  5: bus
  6: train
  7: truck
  8: boat
  9: traffic light
  10: fire hydrant
  11: stop sign
  12: parking meter
  13: bench
  14: bird
  15: cat
  16: dog
  17: horse
  18: sheep
  19: cow
  20: elephant
  21: bear
  22: zebra
  23: giraffe
  24: backpack
  25: umbrella
  26: handbag
  27: tie
  28: suitcase
  29: frisbee
  30: skis
  31: snowboard
  32: sports ball
  33: kite
  34: baseball bat
  35: baseball glove
  36: skateboard
  37: surfboard
  38: tennis racket
  39: bottle
  40: wine glass
  41: cup
  42: fork
  43: knife
  44: spoon
  45: bowl
  46: banana
  47: apple
  48: sandwich
  49: orange
  50: broccoli
  51: carrot
  52: hot dog
  53: pizza
  54: donut
  55: cake
  56: chair
  57: couch
  58: potted plant
  59: bed
  60: dining table
  61: toilet
  62: tv
  63: laptop
  64: mouse
  65: remote
  66: keyboard
  67: cell phone
  68: microwave
  69: oven
  70: toaster
  71: sink
  72: refrigerator
  73: book
  74: clock
  75: vase
  76: scissors
  77: teddy bear
  78: hair drier
  79: toothbrush
"""

with open("/content/drive/MyDrive/2dod/2dod/test_nopatch.yaml", "w") as f:
    f.write(yaml_content)

print("test_nopatch.yaml created.")



In [ ]:
yaml_content = """\
path: /content/drive/MyDrive/2dod/2dod/
train: /content/drive/MyDrive/2dod/2dod/train.txt
val: /content/drive/MyDrive/2dod/2dod/val.txt
test: test_frcnn.txt

names:
  0: person
  1: bicycle
  2: car
  3: motorcycle
  4: airplane
  5: bus
  6: train
  7: truck
  8: boat
  9: traffic light
  10: fire hydrant
  11: stop sign
  12: parking meter
  13: bench
  14: bird
  15: cat
  16: dog
  17: horse
  18: sheep
  19: cow
  20: elephant
  21: bear
  22: zebra
  23: giraffe
  24: backpack
  25: umbrella
  26: handbag
  27: tie
  28: suitcase
  29: frisbee
  30: skis
  31: snowboard
  32: sports ball
  33: kite
  34: baseball bat
  35: baseball glove
  36: skateboard
  37: surfboard
  38: tennis racket
  39: bottle
  40: wine glass
  41: cup
  42: fork
  43: knife
  44: spoon
  45: bowl
  46: banana
  47: apple
  48: sandwich
  49: orange
  50: broccoli
  51: carrot
  52: hot dog
  53: pizza
  54: donut
  55: cake
  56: chair
  57: couch
  58: potted plant
  59: bed
  60: dining table
  61: toilet
  62: tv
  63: laptop
  64: mouse
  65: remote
  66: keyboard
  67: cell phone
  68: microwave
  69: oven
  70: toaster
  71: sink
  72: refrigerator
  73: book
  74: clock
  75: vase
  76: scissors
  77: teddy bear
  78: hair drier
  79: toothbrush
"""

with open("/content/drive/MyDrive/2dod/2dod/test_frcnn.yaml", "w") as f:
    f.write(yaml_content)

print("test_frcnn.yaml created.")


In [ ]:
yaml_content = """\
path: /content/drive/MyDrive/2dod/2dod/
train: /content/drive/MyDrive/2dod/2dod/train.txt
val: /content/drive/MyDrive/2dod/2dod/val.txt
test: test_random.txt

names:
  0: person
  1: bicycle
  2: car
  3: motorcycle
  4: airplane
  5: bus
  6: train
  7: truck
  8: boat
  9: traffic light
  10: fire hydrant
  11: stop sign
  12: parking meter
  13: bench
  14: bird
  15: cat
  16: dog
  17: horse
  18: sheep
  19: cow
  20: elephant
  21: bear
  22: zebra
  23: giraffe
  24: backpack
  25: umbrella
  26: handbag
  27: tie
  28: suitcase
  29: frisbee
  30: skis
  31: snowboard
  32: sports ball
  33: kite
  34: baseball bat
  35: baseball glove
  36: skateboard
  37: surfboard
  38: tennis racket
  39: bottle
  40: wine glass
  41: cup
  42: fork
  43: knife
  44: spoon
  45: bowl
  46: banana
  47: apple
  48: sandwich
  49: orange
  50: broccoli
  51: carrot
  52: hot dog
  53: pizza
  54: donut
  55: cake
  56: chair
  57: couch
  58: potted plant
  59: bed
  60: dining table
  61: toilet
  62: tv
  63: laptop
  64: mouse
  65: remote
  66: keyboard
  67: cell phone
  68: microwave
  69: oven
  70: toaster
  71: sink
  72: refrigerator
  73: book
  74: clock
  75: vase
  76: scissors
  77: teddy bear
  78: hair drier
  79: toothbrush
"""

with open("/content/drive/MyDrive/2dod/2dod/test_random.yaml", "w") as f:
    f.write(yaml_content)

print("test_random.yaml created.")

In [ ]:
yaml_content = """\
path: /content/drive/MyDrive/2dod/2dod/
train: /content/drive/MyDrive/2dod/2dod/train.txt
val: /content/drive/MyDrive/2dod/2dod/val.txt
test: test_retinanet.txt

names:
  0: person
  1: bicycle
  2: car
  3: motorcycle
  4: airplane
  5: bus
  6: train
  7: truck
  8: boat
  9: traffic light
  10: fire hydrant
  11: stop sign
  12: parking meter
  13: bench
  14: bird
  15: cat
  16: dog
  17: horse
  18: sheep
  19: cow
  20: elephant
  21: bear
  22: zebra
  23: giraffe
  24: backpack
  25: umbrella
  26: handbag
  27: tie
  28: suitcase
  29: frisbee
  30: skis
  31: snowboard
  32: sports ball
  33: kite
  34: baseball bat
  35: baseball glove
  36: skateboard
  37: surfboard
  38: tennis racket
  39: bottle
  40: wine glass
  41: cup
  42: fork
  43: knife
  44: spoon
  45: bowl
  46: banana
  47: apple
  48: sandwich
  49: orange
  50: broccoli
  51: carrot
  52: hot dog
  53: pizza
  54: donut
  55: cake
  56: chair
  57: couch
  58: potted plant
  59: bed
  60: dining table
  61: toilet
  62: tv
  63: laptop
  64: mouse
  65: remote
  66: keyboard
  67: cell phone
  68: microwave
  69: oven
  70: toaster
  71: sink
  72: refrigerator
  73: book
  74: clock
  75: vase
  76: scissors
  77: teddy bear
  78: hair drier
  79: toothbrush
"""

with open("/content/drive/MyDrive/2dod/2dod/test_retinanet.yaml", "w") as f:
    f.write(yaml_content)

print("test_retinanet.yaml created.")

In [ ]:
test_sets = {
    "No Patch": "/content/drive/MyDrive/2dod/2dod/test_nopatch.yaml",
    "FRCNN": "/content/drive/MyDrive/2dod/2dod/test_frcnn.yaml",
    "Random": "/content/drive/MyDrive/2dod/2dod/test_random.yaml",
    "RetinaNet": "/content/drive/MyDrive/2dod/2dod/test_retinanet.yaml"
}


In [ ]:
test_results = {}

for name, yaml_path in test_sets.items():
    print(f"Evaluating {name}...")
    results = model.val(data=yaml_path, split="test")

    test_results[name] = {
      "mAP50": results.box.map50,
      "mAP50-95": results.box.map,
      "Precision": results.box.p,
      "Recall": results.box.r
}



In [ ]:
test_results["test_random"] = {
    "mAP50": results.box.map50,
    "mAP75": results.box.map75,     # This is mAP@0.75
    "mAP50-95": results.box.map,    # This is the mean mAP over IoU [.50:.95]
    "Precision": results.box.p,
    "Recall": results.box.r
}

# Print nicely
for k, v in test_results["test_random"].items():
    print(f"{k}: {v:.4f}" if isinstance(v, float) else f"{k}: {v}")


In [ ]:
from pprint import pprint

pprint(values)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

labels = list(test_results.keys())
x = np.arange(len(labels))
width = 0.25

# Extract metrics in order of the labels
mAP50 = [test_results[key]["mAP50"] for key in labels]
mAP90 = [test_results[key]["mAP50-95"] for key in labels]

fig, ax = plt.subplots(figsize=(12, 6))

# Plot bars
bars1 = ax.bar(x - 1.5*width, mAP50, width, label="mAP@50")
bars2 = ax.bar(x - 0.5*width, mAP90, width, label="mAP@50-95")


# Annotate each bar with its value
def annotate_bars(bars):
    for bar in bars:
        height = bar.get_height()
        ax.annotate(f'{height:.2f}',
                    xy=(bar.get_x() + bar.get_width() / 2, height),
                    xytext=(0, 5),  # Offset
                    textcoords="offset points",
                    ha='center', va='bottom',
                    fontsize=9, rotation=0)

for b in [bars1, bars2]:
    annotate_bars(b)

# Styling
ax.set_ylabel("Score")
ax.set_title("Model Performance Across Test Sets")
ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=10)
ax.set_ylim(0, 1.05)
ax.legend(loc='lower center', bbox_to_anchor=(0.5, -0.15), ncol=4)
ax.grid(axis='y', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()
